# Classical ML: decision tree, logistic regression, and one-class SVM

All models run on all five saved splits. Text models share the same TF-IDF settings, and seed 0
is used for the displayed ablation and fitted-model inspection. Logistic regression is run twice
as a **feature ablation**: text only versus text plus the compact EDA-motivated feature set. The tree and
one-class SVM use the same text-plus-structured features.
Hyperparameters are selected with small grouped-CV grids on training rows, thresholds on
validation, and test is evaluated afterward.

In [1]:
import json
import os
import sys
import time
import warnings
from pathlib import Path

if Path.cwd().name == "notebooks":
    os.chdir("..")
sys.path.insert(0, ".")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier

from modules import data_helper as dh
from modules import eval_helper as ev
from modules import explain_helper as eh
from modules import features as ft
from modules import models as md
from modules import report as rp
from modules import results_helper as rh

SEED = 0  # seed used for plots and model inspection
SEEDS = range(5)
OUT = "outputs/classical"
REAL_MODELS = ["logreg_text", "logreg", "tree", "ocsvm"]
RUN_EXPERIMENT = True
N_JOBS = 8  # Adjust to the host; -1 or a positive integer is accepted.

## Data and evaluation split

All preprocessing is fitted by the estimators on training rows (and separately inside each CV fold).
The test rows are displayed here for bookkeeping, not used for feature or parameter selection.

In [2]:
dfe = ft.add_engineered_features(dh.load_processed())
splits = dh.load_splits("splits/seed0.npz", dfe)
train, val, test = (dfe.iloc[splits[name]] for name in ("train", "val", "test"))
y_train = train["fraudulent"].to_numpy()
print({name: {"rows": len(frame), "fraud": int(frame["fraudulent"].sum())}
       for name, frame in (("train", train), ("validation", val), ("test", test))})

{'train': {'rows': 10805, 'fraud': 535}, 'validation': {'rows': 3447, 'fraud': 151}, 'test': {'rows': 3345, 'fraud': 170}}


## Models and small search grids

No model reweights classes: AP ranks scores and every threshold is chosen afterwards on validation F1,
so class weighting is not needed for either; the DistilBERT models follow the same rule.

- **Dummy:** majority class; no training parameters. It is the floor every model must beat.
- **Logistic regression:** the supervised linear reference, tuning only the regularization strength `C`.
  It runs twice, text only and text plus structured features: this pair is the classical feature ablation.
- **Decision tree:** a nonlinear contrast on the same text-plus-structured features as structured LR;
  `max_depth` and `min_samples_leaf` control overfitting.
- **One-class SVM:** an anomaly-detection contrast that never sees fraud labels when fitting. It uses the
  same features as structured LR, reduced to 100 PCA components and unit-length rows before an RBF
  kernel, because RBF distances are poorly behaved in the raw 20,000-dimensional sparse space. All
  stages are fitted on known-real training rows only.

In [3]:
from sklearn.model_selection import GridSearchCV, ParameterGrid, StratifiedGroupKFold

print("models:", list(md.get_models(SEED)))
print("grid sizes:", {name: len(list(ParameterGrid(grid)))
                      for name, grid in md.PARAM_GRIDS.items()})

models: ['logreg_text', 'logreg', 'tree', 'ocsvm']
grid sizes: {'logreg_text': 6, 'logreg': 6, 'tree': 12, 'ocsvm': 10}


## Run the experiments

Average precision (AP) is the main metric for this imbalanced task. Each non-dummy model uses an
exhaustive small grid with three-fold grouped CV on training. The output folder is reused only when
both files for a finished run exist; use a new output folder after changing the pipeline.

In [4]:
if RUN_EXPERIMENT:
    for experiment_seed in SEEDS:
        dh.seed_everything(experiment_seed)
        run_splits = dh.load_splits(f"splits/seed{experiment_seed}.npz", dfe)
        fit_train, fit_val, fit_test = (
            dfe.iloc[run_splits[name]] for name in ("train", "val", "test")
        )
        y_train, y_val, y_test = (
            part["fraudulent"].to_numpy() for part in (fit_train, fit_val, fit_test)
        )
        print(f"seed {experiment_seed}: rows train/val/test = "
              f"{[len(fit_train), len(fit_val), len(fit_test)]}, positives = "
              f"{[int(y_train.sum()), int(y_val.sum()), int(y_test.sum())]}", flush=True)
        for name in ["dummy_majority", *REAL_MODELS]:
            if rh.result_exists(OUT, name, experiment_seed, fingerprint=dh.data_fingerprint(dfe),
                                val_indices=run_splits["val"], test_indices=run_splits["test"]):
                print(f"seed {experiment_seed} {name}: already done, skipping")
                continue

            start = time.time()
            with warnings.catch_warnings(record=True) as caught:
                warnings.simplefilter("always")
                if name == "dummy_majority":
                    model = DummyClassifier(strategy="most_frequent", random_state=experiment_seed)
                    model.fit(fit_train, y_train)
                    best_params, cv_score = {}, None
                else:
                    folds = StratifiedGroupKFold(
                        n_splits=3, shuffle=True, random_state=experiment_seed
                    )
                    search = GridSearchCV(
                        md.get_models(experiment_seed)[name],
                        md.PARAM_GRIDS[name],
                        scoring="average_precision",
                        cv=folds,
                        n_jobs=N_JOBS,
                        error_score="raise",
                    )
                    search.fit(fit_train, y_train, groups=fit_train["group"].to_numpy())
                    model = search.best_estimator_
                    best_params = search.best_params_
                    cv_score = float(search.best_score_)
            warned = sorted({str(w.message).splitlines()[0][:80] for w in caught})

            score_train, score_val, score_test = (
                ev.score_of(model, part) for part in (fit_train, fit_val, fit_test)
            )
            threshold = 0.5 if name == "dummy_majority" else ev.pick_threshold(y_val, score_val)
            val_metrics = ev.evaluate(y_val, score_val, threshold)
            test_metrics = ev.evaluate(y_test, score_test, threshold)
            seconds = round(time.time() - start, 1)

            if name == "dummy_majority":
                search_desc = "not tuned"
            else:
                candidates = len(ParameterGrid(md.PARAM_GRIDS[name]))
                search_desc = f"{candidates}-candidate grid, 3-fold grouped CV; CV AP {cv_score:.4f}"
            rh.save_result(
                OUT,
                name,
                "classical",
                experiment_seed,
                threshold,
                val_metrics,
                test_metrics,
                y_val,
                score_val,
                y_test,
                score_test,
                train_pr_auc=ev.evaluate(y_train, score_train, threshold)["pr_auc"],
                best_params=best_params,
                seconds=seconds,
                notes=f"{search_desc}; warnings: {warned or 'none'}",
                data_fingerprint=dh.data_fingerprint(dfe),
                val_indices=run_splits["val"], test_indices=run_splits["test"],
            )
            print(
                f"seed {experiment_seed} {name:16s} val AP {val_metrics['pr_auc']:.3f} "
                f"F1 {val_metrics['f1']:.3f} | {seconds}s"
                + (f" | WARNINGS: {'; '.join(warned)}" if warned else ""),
                flush=True,
            )

seed 0: rows train/val/test = [10805, 3447, 3345], positives = [535, 151, 170]


seed 0 dummy_majority: already done, skipping


seed 0 logreg_text      val AP 0.653 F1 0.662 | 46.2s


seed 0 logreg           val AP 0.622 F1 0.609 | 51.8s


seed 0 tree: already done, skipping
seed 0 ocsvm: already done, skipping


seed 1: rows train/val/test = [10314, 3564, 3719], positives = [500, 175, 181]


... (14 outputs left out)

seed 4 dummy_majority: already done, skipping


seed 4 logreg_text      val AP 0.798 F1 0.765 | 53.1s


seed 4 logreg           val AP 0.822 F1 0.795 | 65.8s


seed 4 tree: already done, skipping
seed 4 ocsvm: already done, skipping


In [5]:
results = rh.load_results(OUT)
results = results[results["seed"] == SEED].copy()
assert set(results["model"]) == {"dummy_majority", *REAL_MODELS}
metrics = ["pr_auc", "precision", "recall", "f1"]
comparison = results.set_index("model")[[f"{split}_{metric}" for split in ("val", "test")
                                        for metric in metrics]]
comparison.round(3)

,val_pr_auc,val_precision,val_recall,val_f1,test_pr_auc,test_precision,test_recall,test_f1
model,,,,,,,,
dummy_majority,0.044,0.000,0.000,0.000,0.051,0.000,0.000,0.000
logreg,0.622,0.804,0.490,0.609,0.749,0.924,0.571,0.705
logreg_text,0.653,0.777,0.576,0.662,0.659,0.772,0.518,0.620
ocsvm,0.051,0.065,0.490,0.115,0.041,0.028,0.259,0.051
tree,0.324,0.560,0.371,0.446,0.523,0.790,0.553,0.651


In [6]:
records = {}
for name in ["dummy_majority", *REAL_MODELS]:
    with open(f"{OUT}/{name}_seed{SEED}.json", encoding="utf-8") as f:
        records[name] = json.load(f)
pd.Series({name: str(record["best_params"]) for name, record in records.items()},
          name="selected parameters").to_frame()

,selected parameters
dummy_majority,{}
logreg_text,{'clf__C': 100}
logreg,{'clf__C': 1000}
tree,"{'clf__max_depth': 10, 'clf__min_samples_leaf'..."
ocsvm,"{'detector__svm__gamma': 30.0, 'detector__svm_..."


## Does structured feature engineering help logistic regression?

`logreg_text` and `logreg` differ only in the structured feature blocks. Positive differences favour
adding the EDA-motivated features. This comparison is descriptive for the seed-0 split.

In [7]:
delta = comparison.loc["logreg"] - comparison.loc["logreg_text"]
delta.rename("structured minus text-only").round(3).to_frame()

,structured minus text-only
val_pr_auc,-0.031
val_precision,0.028
val_recall,-0.086
val_f1,-0.053
test_pr_auc,0.090
test_precision,0.152
test_recall,0.053
test_f1,0.086


## Inspect the fitted supervised models

Refit only on training rows using the saved parameters, solely to recover fitted coefficients and
importances. No new metric is selected from test data.

In [8]:
def refit_for_explanation(name):
    model = md.get_models(SEED)[name].set_params(**records[name]["best_params"])
    return model.fit(train, train["fraudulent"])


logreg = refit_for_explanation("logreg")
eh.top_linear_terms(logreg, k=15)

,term,coef,direction
0,text__link url,14.448032,positive
1,text__timejob,10.721794,positive
2,text__money,10.513279,positive
3,text__northwestern,10.244669,positive
4,text__sales executive,9.700160,positive
5,text__link,9.611547,positive
6,text__url,9.213369,positive
7,text__club,8.712488,positive
8,text__real,8.523376,positive
9,text__novation,8.115789,positive


In [9]:
tree = refit_for_explanation("tree")
importances = pd.Series(
    tree.named_steps["clf"].feature_importances_,
    index=tree.named_steps["pre"].get_feature_names_out(),
)
importances.nlargest(15).rename("importance").to_frame()

,importance
text__leveraging career,0.199041
binary__has_company_logo,0.126608
text__data entry,0.097310
text__free,0.065574
text__link url,0.064024
text__facilitating,0.061645
text__cell phones,0.058164
nominal__industry_Oil & Energy,0.039395
nominal__function_Administrative,0.032786
text__desire,0.027806


In [10]:
def feature_label(term):
    """Readable report label for a fitted feature name such as 'nominal__industry_Oil & Energy'."""
    block, _, value = term.partition("__")
    if block == "nominal":
        for field in ft.NOMINAL_COLS:
            if value.startswith(field + "_"):
                return f"{field.replace('_', ' ')}: {value[len(field) + 1:]}"
    return value.replace("_", " ")


# Report tables: ten largest coefficients in each direction, and the ten largest tree importances.
terms = eh.top_linear_terms(logreg, k=10)
positive = terms[terms["direction"] == "positive"].reset_index(drop=True)
negative = terms[terms["direction"] == "negative"].reset_index(drop=True)
rp.write_table_rows("tab_terms.tex", [
    (rp.tex(feature_label(p.term)), f"{p.coef:+.2f}", rp.tex(feature_label(n.term)), f"{n.coef:+.2f}")
    for p, n in zip(positive.itertuples(), negative.itertuples())
])
rp.write_table_rows("tab_tree_terms.tex", [
    (rp.tex(feature_label(name)), f"{value:.3f}") for name, value in importances.nlargest(10).items()
])

saved report table tab_terms.tex
saved report table tab_tree_terms.tex


In [11]:
scores = {"logreg": rh.load_scores(OUT, "logreg", SEED)}
y_error = np.asarray(scores["logreg"]["y_val"]).astype(int)
score_error = np.asarray(scores["logreg"]["score_val"], dtype=float)
pred_error = score_error >= records["logreg"]["thr"]
fp_idx = np.where((pred_error == 1) & (y_error == 0))[0]
fn_idx = np.where((pred_error == 0) & (y_error == 1))[0]
fp_idx = fp_idx[np.argsort(-score_error[fp_idx])][:3]
fn_idx = fn_idx[np.argsort(score_error[fn_idx])][:3]
error_rows = []
for kind, indices in (("false_positive", fp_idx), ("false_negative", fn_idx)):
    for i in indices:
        row = val.iloc[i]
        error_rows.append({
            "error_type": kind,
            "score": float(score_error[i]),
            "y_true": int(y_error[i]),
            "title": str(row["title"]),
            "description": str(row["description"])[:200],
        })
for error in error_rows:
    print(f"[{error['error_type']}] score={error['score']:.3f} | {error['title']}")
    print(error["description"], "\n")

[false_positive] score=1.000 | Data Entry
This is a data entry position, Working hours are flexible. you are also expected to come online through Yahoo Messenger during working hours and also for your online interview / job briefing.i will li 

[false_positive] score=0.934 | Controls - PLC Programmer
CONTROLS – PLC PROGRAMMEREDMONTON, ALBERTAPackers Plus is the pioneer of open hole, multi-stage horizontal fracturing and we provide technology-based solutions to the oil and gas industry around the w 

[false_positive] score=0.931 | Director of Program Management - Six Sigma Lean - PA
D - 124Experienced Director of Program Management is required having responsibility to monitor supply chain in order to develop processes to optimize concurrent engineering projects  Job Responsibilit 

[false_negative] score=0.000 | Controls Engineer - Farmington Hills, MI
Title: Controls EngineerDuration: PermanentLocation: Farmington Hills, MI JOB DESCRIPTION:Controls Engineer Position-PLC programming and

## Seed-0 audit conclusion

On this split structured LR has the best test AP (0.749 against 0.659 for text-only LR), but validation
disagrees: text-only LR is better on validation AP (0.653 against 0.622). One split cannot settle the
feature ablation, which is why notebook 04 compares all five paired splits. Both LR variants fit the
training rows almost perfectly (train AP 1.0 against grouped-CV AP about 0.50–0.52), so their ranking
quality on new ads is limited by generalization, not capacity.

The decision tree reaches 0.523 test AP, below both LR variants. The one-class SVM, fitted without fraud
labels, scores 0.041 test AP, below the 0.051 fraud rate of this test split: unusual ads are not
fraudulent ads here. The printed errors show false alarms on a remote data-entry ad and on oil-and-gas and Six Sigma
engineering ads, and confident misses on one templated series of controls-engineer ads; individual
examples alone do not establish label errors.

**Next:** notebook 03 fine-tunes DistilBERT on the same five splits, and notebook 04 compares every
model with paired differences.